# Air Quality Forecasting and Pollution Analytics

**Author:** Mandeep Kumar  
**Dataset:** CPCB India — city_day.csv (29,531 records)  

This notebook walks through the complete analysis pipeline interactively:
1. Data loading & preprocessing
2. Exploratory data analysis
3. Feature engineering
4. Regression modelling (AQI prediction)
5. Classification modelling (AQI Bucket)
6. Model evaluation & visualisations

In [20]:
import sys
from pathlib import Path

# Add src/ to path so we can import project modules
# Robustly find project root by searching for marker files or a parent with 'src'
p = Path().resolve()
ROOT = None
for _ in range(20):
    if (p / 'requirements.txt').exists() or (p / 'run_pipeline.py').exists():
        ROOT = p
        break
    if p.parent == p:
        break
    p = p.parent
if ROOT is None:
    # fallback: search upward from current working directory for a folder containing 'src'
    q = Path.cwd()
    for _ in range(20):
        if (q / 'src').exists():
            ROOT = q
            break
        if q.parent == q:
            break
        q = q.parent
if ROOT is None:
    ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

print('Libraries loaded successfully')
print('Detected ROOT =', ROOT)
print('Inserted to sys.path =', sys.path[0])

Libraries loaded successfully
Detected ROOT = /content
Inserted to sys.path = /content/src


In [25]:
import sys, os
print('ROOT =', ROOT)
p = str((ROOT if ROOT is not None else Path.cwd()) / 'src')
print('expected src path =', p)
print('sys.path[0] =', sys.path[0])
print('exists?', os.path.exists(sys.path[0]))
print('listing of sys.path[0] (first 20):', os.listdir(sys.path[0])[:20])

ROOT = /content
expected src path = /content/src
sys.path[0] = /content/src
exists? False


FileNotFoundError: [Errno 2] No such file or directory: '/content/src'

## 1. Data Loading & Preprocessing

In [ ]:
from data_preprocessing import preprocess

df = preprocess()
print(f'Cleaned dataset shape: {df.shape}')
df.head()

2026-05-17 22:57:33 | INFO     | data_preprocessing | Loading raw data from D:\air-quality-forecasting-and-pollution-analytics\data\city_day.csv


2026-05-17 22:57:33 | INFO     | data_preprocessing | Raw shape: (29531, 16)


2026-05-17 22:57:33 | INFO     | data_preprocessing | Dropped 4681 rows with missing AQI/AQI_Bucket (29531 → 24850)


2026-05-17 22:57:33 | INFO     | data_preprocessing | Imputed 43525 missing numeric values with column medians (remaining: 0)


2026-05-17 22:57:33 | INFO     | data_preprocessing | Removed 0 duplicate rows (24850 → 24850)


2026-05-17 22:57:34 | INFO     | data_preprocessing | Cleaned dataset saved to D:\air-quality-forecasting-and-pollution-analytics\data\cleaned_air_quality.csv  (shape: (24850, 16))


Cleaned dataset shape: (24850, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-29,83.13,96.18,6.93,28.71,33.72,16.31,6.93,49.52,59.76,0.02,0.00,3.14,209.0,Poor
1,Ahmedabad,2015-01-30,79.84,96.18,13.85,28.68,41.08,16.31,13.85,48.49,97.07,0.04,0.00,4.81,328.0,Very Poor
2,Ahmedabad,2015-01-31,94.52,96.18,24.39,32.66,52.61,16.31,24.39,67.39,111.33,0.24,0.01,7.67,514.0,Severe
3,Ahmedabad,2015-02-01,135.99,96.18,43.48,42.08,84.57,16.31,43.48,75.23,102.70,0.40,0.04,25.87,782.0,Severe
4,Ahmedabad,2015-02-02,178.33,96.18,54.56,35.31,72.80,16.31,54.56,55.04,107.38,0.46,0.06,35.61,914.0,Severe


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24850 entries, 0 to 24849
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   City        24850 non-null  object        
 1   Date        24850 non-null  datetime64[ns]
 2   PM2.5       24850 non-null  float64       
 3   PM10        24850 non-null  float64       
 4   NO          24850 non-null  float64       
 5   NO2         24850 non-null  float64       
 6   NOx         24850 non-null  float64       
 7   NH3         24850 non-null  float64       
 8   CO          24850 non-null  float64       
 9   SO2         24850 non-null  float64       
 10  O3          24850 non-null  float64       
 11  Benzene     24850 non-null  float64       
 12  Toluene     24850 non-null  float64       
 13  Xylene      24850 non-null  float64       
 14  AQI         24850 non-null  float64       
 15  AQI_Bucket  24850 non-null  object        
dtypes: datetime64[ns](1), 

In [ ]:
df.describe().round(2)

,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI
count,24850,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00,24850.00
mean,2018-07-24 18:51:25.714285568,66.97,112.10,17.50,28.87,31.65,21.87,2.32,14.24,34.79,3.15,8.13,2.25,166.46
min,2015-01-01 00:00:00,0.04,0.03,0.03,0.01,0.00,0.01,0.00,0.01,0.01,0.00,0.00,0.00,13.00
25%,2017-08-16 00:00:00,29.56,71.78,5.72,12.09,14.03,11.28,0.59,5.79,19.64,0.34,1.58,1.42,81.00
50%,2018-11-05 00:00:00,48.78,96.18,9.91,22.10,23.68,16.31,0.93,9.22,31.25,1.29,3.58,1.42,118.00
75%,2019-10-11 00:00:00,79.51,122.96,19.71,37.91,38.17,24.71,1.46,14.89,45.40,2.85,7.38,1.42,208.00
max,2020-07-01 00:00:00,914.94,917.08,390.68,362.21,378.24,352.89,175.81,186.08,257.73,455.03,454.85,170.37,2049.00
std,NaN,62.28,76.33,22.27,24.45,29.63,22.46,7.01,17.23,21.38,14.87,18.44,4.30,140.70


In [ ]:
print('Missing values after cleaning:')
print(df.isnull().sum())

Missing values after cleaning:
City          0
Date          0
PM2.5         0
PM10          0
NO            0
NO2           0
NOx           0
NH3           0
CO            0
SO2           0
O3            0
Benzene       0
Toluene       0
Xylene        0
AQI           0
AQI_Bucket    0
dtype: int64


## 2. Exploratory Data Analysis

In [ ]:
# AQI Distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['AQI'], bins=60, color='#3498db', edgecolor='white', alpha=0.8)
ax.set_xlabel('AQI Value')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Air Quality Index (AQI)', fontweight='bold')
plt.tight_layout()
plt.show()

C:\Users\Mandeep\AppData\Local\Temp\ipykernel_34188\4132565224.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# AQI Bucket counts
bucket_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
bucket_colors = ['#2ecc71', '#a8e063', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
present = [b for b in bucket_order if b in df['AQI_Bucket'].values]
counts = df['AQI_Bucket'].value_counts().reindex(present)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(present, counts.values, color=bucket_colors[:len(present)], edgecolor='white')
ax.set_xlabel('AQI Bucket')
ax.set_ylabel('Count')
ax.set_title('AQI Bucket Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

C:\Users\Mandeep\AppData\Local\Temp\ipykernel_34188\3695954946.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Top 15 most polluted cities
city_aqi = df.groupby('City')['AQI'].mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(city_aqi.index[::-1], city_aqi.values[::-1], color='#e74c3c', edgecolor='white')
ax.set_xlabel('Mean AQI')
ax.set_title('Top 15 Most Polluted Cities', fontweight='bold')
plt.tight_layout()
plt.show()

C:\Users\Mandeep\AppData\Local\Temp\ipykernel_34188\2080598277.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Monthly AQI trend
df['Month'] = pd.to_datetime(df['Date']).dt.month
monthly = df.groupby('Month')['AQI'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly.index, monthly.values, marker='o', linewidth=2.5, color='#e74c3c')
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='#e74c3c')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_xlabel('Month')
ax.set_ylabel('Mean AQI')
ax.set_title('Monthly Average AQI Trend', fontweight='bold')
plt.tight_layout()
plt.show()

C:\Users\Mandeep\AppData\Local\Temp\ipykernel_34188\4030652801.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Correlation heatmap
pollutant_cols = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene','Xylene','AQI']
corr = df[pollutant_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_xticks(range(len(pollutant_cols)))
ax.set_yticks(range(len(pollutant_cols)))
ax.set_xticklabels(pollutant_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(pollutant_cols, fontsize=9)
ax.set_title('Correlation Heatmap', fontweight='bold')
for i in range(len(pollutant_cols)):
    for j in range(len(pollutant_cols)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha='center', va='center', fontsize=7)
plt.tight_layout()
plt.show()

C:\Users\Mandeep\AppData\Local\Temp\ipykernel_34188\2943710599.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Feature Engineering

In [ ]:
from feature_engineering import build_feature_matrix

X, y_reg, y_clf = build_feature_matrix(df)
print('Feature matrix shape:', X.shape)
print('Features:', list(X.columns))
print('\nAQI range:', y_reg.min(), '–', y_reg.max())
print('AQI buckets:', y_clf.unique())

2026-05-17 22:57:36 | INFO     | feature_engineering | Temporal features added: ['Year', 'Month', 'Day', 'DayOfWeek', 'Quarter']


2026-05-17 22:57:36 | INFO     | feature_engineering | City encoded: 26 unique cities


2026-05-17 22:57:36 | INFO     | feature_engineering | Feature matrix built: X=(24850, 18)  |  y_reg=(24850,)  |  y_clf=(24850,)


Feature matrix shape: (24850, 18)
Features: ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'City_Encoded']

AQI range: 13.0 – 2049.0
AQI buckets: ['Poor' 'Very Poor' 'Severe' 'Moderate' 'Satisfactory' 'Good']


## 4. Regression Modelling

In [ ]:
from train_regression import train_regression

best_reg_model, reg_results, best_reg_name = train_regression(X, y_reg)

print('\nRegression Results:')
for name, m in reg_results.items():
    marker = ' ← BEST' if name == best_reg_name else ''
    print(f'  {name}: MAE={m["MAE"]}  RMSE={m["RMSE"]}  R²={m["R2"]}{marker}')

2026-05-17 22:57:36 | INFO     | train_regression | Train/test split: 19880 / 4970


2026-05-17 22:57:36 | INFO     | train_regression | Training Linear Regression …


2026-05-17 22:57:37 | INFO     | train_regression |   Linear Regression → MAE=30.9888  RMSE=59.2599  R²=0.8082


2026-05-17 22:57:37 | INFO     | train_regression | Training Random Forest Regressor …


2026-05-17 22:57:42 | INFO     | train_regression |   Random Forest Regressor → MAE=20.7132  RMSE=40.6391  R²=0.9098


...

2026-05-17 22:58:08 | INFO     | train_regression |   Gradient Boosting Regressor → MAE=20.9055  RMSE=39.6147  R²=0.9143


2026-05-17 22:58:08 | INFO     | train_regression | Best regression model: Gradient Boosting Regressor  (R²=0.9143)


2026-05-17 22:58:08 | INFO     | train_regression | Saved best regression model → D:\air-quality-forecasting-and-pollution-analytics\models\aqi_regression_model.pkl


2026-05-17 22:58:08 | INFO     | train_regression | Saved preprocessor (scaler) → D:\air-quality-forecasting-and-pollution-analytics\models\preprocessor.pkl



Regression Results:
  Linear Regression: MAE=30.9888  RMSE=59.2599  R²=0.8082
  Random Forest Regressor: MAE=20.7132  RMSE=40.6391  R²=0.9098
  Gradient Boosting Regressor: MAE=20.9055  RMSE=39.6147  R²=0.9143 ← BEST


## 5. Classification Modelling

In [ ]:
from train_classification import train_classification

best_clf_model, clf_results, best_clf_name, le, y_test_enc = train_classification(X, y_clf)

print('\nClassification Results:')
for name, m in clf_results.items():
    marker = ' ← BEST' if name == best_clf_name else ''
    print(f'  {name}: Acc={m["Accuracy"]}  F1={m["F1_Score"]}{marker}')

2026-05-17 22:58:08 | INFO     | train_classification | Train/test split: 19880 / 4970


2026-05-17 22:58:08 | INFO     | train_classification | Training Logistic Regression …


2026-05-17 22:58:10 | INFO     | train_classification |   Logistic Regression → Acc=0.7531  P=0.7509  R=0.7531  F1=0.7456


2026-05-17 22:58:10 | INFO     | train_classification | Training Random Forest Classifier …


2026-05-17 22:58:12 | INFO     | train_classification |   Random Forest Classifier → Acc=0.8165  P=0.8159  R=0.8165  F1=0.8143


2026-05-17 22:58:12 | INFO     | train_classification | Training Gradient Boosting Classifier …


2026-05-17 23:01:14 | INFO     | train_classification |   Gradient Boosting Classifier → Acc=0.8070  P=0.8059  R=0.8070  F1=0.8062


2026-05-17 23:01:14 | INFO     | train_classification | Best classifier: Random Forest Classifier  (F1=0.8143)


2026-05-17 23:01:14 | INFO     | train_classification | Saved best classifier → D:\air-quality-forecasting-and-pollution-analytics\models\aqi_bucket_classifier.pkl



Classification Results:
  Logistic Regression: Acc=0.7531  F1=0.7456
  Random Forest Classifier: Acc=0.8165  F1=0.8143 ← BEST
  Gradient Boosting Classifier: Acc=0.807  F1=0.8062


## 6. Feature Importance

In [ ]:
from feature_engineering import FEATURE_COLS

reg_step = best_reg_model.named_steps['model']
if hasattr(reg_step, 'feature_importances_'):
    importances = reg_step.feature_importances_
    idx = np.argsort(importances)[::-1][:15]
    top_features = [FEATURE_COLS[i] for i in idx]
    top_values = importances[idx]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_features[::-1], top_values[::-1], color='#3498db', edgecolor='white')
    ax.set_xlabel('Importance Score')
    ax.set_title(f'Feature Importance — {best_reg_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()

C:\Users\Mandeep\AppData\Local\Temp\ipykernel_34188\803227430.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Confusion Matrix

In [ ]:
import numpy as np

cm = np.array(clf_results[best_clf_name]['Confusion_Matrix'])
class_labels = list(le.classes_)
n = len(class_labels)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(class_labels, rotation=45, ha='right')
ax.set_yticklabels(class_labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {best_clf_name}', fontweight='bold')
thresh = cm.max() / 2
for i in range(n):
    for j in range(n):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')
plt.tight_layout()
plt.show()

C:\Users\Mandeep\AppData\Local\Temp\ipykernel_34188\2152911002.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

- **Best regression model:** see `reg_results` above
- **Best classification model:** see `clf_results` above
- All models, metrics, and plots are saved by `run_pipeline.py`

Run `python run_pipeline.py` from the project root to regenerate all outputs.